In [ ]:
# Common
import keras
import numpy as np
import tensorflow as tf
from tensorflow import image as tfi

# Data
from keras.datasets import cifar100

# Data Viz
import matplotlib.pyplot as plt

# Data Aug
from keras.layers import Normalization
from keras.layers import Resizing
from keras.layers import RandomFlip
from keras.layers import RandomRotation
from keras.layers import RandomZoom


# Model
from tensorflow.nn import gelu
from keras.models import Model
from keras.layers import Dense
from keras.layers import Layer
from keras.layers import Input
from keras.layers import Embedding
from keras.layers import Dropout
from keras.layers import MultiHeadAttention
from keras.layers import LayerNormalization
from keras.layers import Add
from keras.layers import Flatten
from tensorflow.keras.optimizers import Adam

# Compiling
from keras.losses import SparseCategoricalCrossentropy as SCCe
from tensorflow_addons.optimizers import AdamW
from keras.metrics import SparseCategoricalAccuracy as Acc
from keras.metrics import SparseTopKCategoricalAccuracy as KAcc

# Callbacks 
from keras.callbacks import ModelCheckpoint
from keras.callbacks import EarlyStopping


In [ ]:
from keras.preprocessing.image import ImageDataGenerator

BATCH_SIZE=32

SEED=1345

train_datagen=ImageDataGenerator(rescale=1./255)

validation_datagen=ImageDataGenerator(rescale=1./255)
test_datagen=ImageDataGenerator(rescale=1./255)


#Defining directories for train,validation,test 
train_dir = 'Splitted2D/train'
validation_dir = 'Splitted2D/val'
test_dir = 'Splitted2D/test'


#Defining generatores for train,validation,test

train_generator=train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    shuffle=True,
    batch_size= 32,
    class_mode ='categorical',
)

validation_generator = validation_datagen.flow_from_directory(
    validation_dir,
    target_size=(224, 224),
    shuffle=True,
    batch_size= 32,
    class_mode ='categorical',
)

# Define generator for test set using flow_from_directory
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    shuffle=True,
    batch_size = 32,
    class_mode ='categorical',
)

In [ ]:
x_train, y_train = next(train_generator)
x_val, y_val = next(validation_generator)
x_test, y_test = next(test_generator)

In [ ]:
input_shape = (224, 224, 3)
SIZE = 224
PATCH_SIZE = 16
LR = 0.001
WEIGHT_DECAY = 0.0001
EPOCHS = 100
NUM_PATCHES = (SIZE // PATCH_SIZE) ** 2
PROJECTION_DIMS = 512
NUM_HEADS = 16
HIDDEN_UNITS = [PROJECTION_DIMS*2, PROJECTION_DIMS]
OUTPUT_UNITS = [2048,1024]

In [ ]:
class DataAugmentation(Layer):
    def __init__(self, norm, SIZE):
        super(DataAugmentation, self).__init__()
        self.norm = norm
        self.SIZE = SIZE
        self.resize = Resizing(SIZE, SIZE)
        self.rotation = RandomRotation(factor=0.02)
        self.zoom = RandomZoom(height_factor=0.2, width_factor=0.2)
    def call(self, X):
        x = self.norm(X)
        x = self.resize(x)
        x = self.rotation(x)
        x = self.zoom(x)
        return x
    
    def get_config(self):
        config = super().get_config()
        config.update({
        "norm": self.norm,
        "SIZE": self.SIZE,
        })
        return config


In [ ]:
class Patches(Layer):

    def __init__(self, patch_size):
        super(Patches, self).__init__()
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0] # Get the Batch Size
        patches = tfi.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1], # only along the Height and Width Dimension
            strides=[1, self.patch_size, self.patch_size, 1], # The next patch should not overlap the previus patch
            rates=[1,1,1,1],
            padding='VALID'
        )
        patch_dims = patches.shape[-1]
        patches = tf.reshape(patches, [batch_size, -1, patch_dims])
        return patches
    def get_config(self):
        config = super().get_config()
        config.update({
            "path-size": self.patch_size,
        })
        return config


In [ ]:
class PatchEncoder(Layer):

    def __init__(self, num_patches, projection_dims): # Projection dims is  D
        super(PatchEncoder, self).__init__()
        self.num_patches = num_patches
        self.d = projection_dims

        self.dense = Dense(units=projection_dims)
        self.positional_embeddings = Embedding(input_dim=num_patches, output_dim=projection_dims)

    def call(self, X):
        positions = tf.range(0,limit=self.num_patches, delta=1)
        encoded = self.dense(X) + self.positional_embeddings(positions)
        return encoded
    def get_config(self):
        config = super().get_config()
        config.update({
            "num_paches": self.num_patches,
            "d": self.d,
        })
        return config

In [ ]:
class MLP(Layer):
    def __init__(self, units, rate):
        super(MLP, self).__init__()
        self.units = units
        self.rate = rate
        self.layers = [[Dense(unit, activation=gelu), Dropout(rate)] for unit in units]

    def call(self, x):
        for layers in self.layers:
            for layer in layers:
                x = layer(x)
        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            "units": self.units,
            "rate": self.rate,
        })
        return config


In [ ]:
class Transformer(Layer):
    # L means number of blocks
    def __init__(self, L, num_heads, key_dims, hidden_units):
        super(Transformer, self).__init__()
        self.L = L
        self.heads = num_heads
        self.key_dims = key_dims
        self.hidden_units = hidden_units

        self.norm = LayerNormalization(epsilon=1e-6) # Remember the Params
        self.MHA = MultiHeadAttention(num_heads=num_heads, key_dim=key_dims, dropout=0.1)
        self.net = MLP(units=hidden_units, rate=0.1)
        self.add= Add()

    def call(self, X):
        inputs = X
        x = X
        for _ in range(self.L):
            x = self.norm(x)
            x = self.MHA(x,x) # our Target and the Source element are the same
            y = self.add([x,inputs])
            x = self.norm(y)
            x = self.net(x)
            x = self.add([x,y])
        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            "L": self.L,
            "heads": self.heads,
            "key_dims":self.key_dims,
            "hidden_units":self.hidden_units
        })
        return config

In [ ]:
with tf.device('/cpu:0'):
    inputs = Input(shape=input_shape)

    # Apply Data Aug
    norm = Normalization()
    norm.adapt(x_train)

    x = DataAugmentation(norm, SIZE)(inputs)

    # Get Patches
    x = Patches(PATCH_SIZE)(x)

    # PatchEncoding Network
    x = PatchEncoder(NUM_PATCHES, PROJECTION_DIMS)(x)

    # Transformer Network
    x = Transformer(8, NUM_HEADS, PROJECTION_DIMS, HIDDEN_UNITS)(x)

    # Output Network
    x = LayerNormalization(epsilon=1e-6)(x)
    x = Flatten()(x)
    x = Dropout(0.5)(x)

    x = MLP(OUTPUT_UNITS, rate=0.5)(x)

    # Ouput Layer
    outputs = Dense(4, activation = 'softmax')(x)

In [ ]:
with tf.device('/cpu:0'):
    # Model
    model = Model(
        inputs=[inputs],
        outputs=[outputs],
    )

    # Compiling
    model.compile(optimizer=Adam(learning_rate = LR, decay = WEIGHT_DECAY),
                  loss='categorical_crossentropy', metrics=['accuracy',tf.keras.metrics.AUC(),
                            tf.keras.metrics.Precision(),
                            tf.keras.metrics.Recall(),])

    # Fit
    results = model.fit(
        train_generator,
        epochs=EPOCHS,
        validation_data=validation_generator,
        steps_per_epoch = len(train_generator),
        verbose = 1
    )
